# Обучение и инференс тела модели CDS

## Подготовка данных

In [ ]:
!bash /home/datalab/nfs/amazmefmlib/load_data/sequence_representation_8M.sh

## Импорты и проверка работоспособности кластера

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("/opt/clients"))

import osiris

In [2]:
osiris.list()

{'jobs': [{'job': 'training-🔐🔐🔐🔐🔐🔐🔐🔐-pytorchjob-accelerate-fmlib-2x4',
   'uuid': '9e176e85-df63-4c4a-bfa8-cb668ba28d2c',
   'state': 'Running',
   'transition': '2026-02-05T14:36:14Z',
   'creation': '2026-02-05T14:36:02Z'}],
 'time': '2026-02-05T14:36:29Z'}

In [3]:
for job in osiris.list()["jobs"]:
    osiris.delete(job["job"])

## Запуск обучения модели

In [4]:
job = osiris.create(
    name="accelerate-fmlib-2x4",
    image="registry.ca.sbrf.ru/ci02684173/ci02697916/notebooks/python3.12/cuda12.4/d-03.000.00:d-03.000.00-gigachat",
    restart=False,
    pool="public",
    command=[
        "/home/datalab/nfs/amazmefmlib_hgx_venv/bin/accelerate",
        "launch",
    ],
    args=[
        "--mixed_precision",
        "no",
        "--dynamo_backend",
        "no",
        "--num_machines",
        "2",
        "--num_processes",
        "8",
        "--main_process_ip",
        "$(MASTER_ADDR).datalab.svc.cluster.local",
        "--main_process_port",
        "$(MASTER_PORT)",
        "--machine_rank",
        "$(RANK)",
        "/home/datalab/nfs/amazmefmlib/fmlib/training/train.py",
        "--config-dir=/home/datalab/nfs/amazmefmlib/examples/configs/train",
        "--config-name=sequence_representation_2M",
    ],
    envs={
        "PYTHONPATH": "/home/datalab/nfs/amazmefmlib_hgx_venv",
        "OMP_NUM_THREADS": "24",
        "NCCL_DEBUG": "INFO",
    },
    num_nodes=2,
    num_gpus=4,
    type="pytorchjob",
)
job

{'job': 'training-🔐🔐🔐🔐🔐🔐🔐🔐-pytorchjob-accelerate-fmlib-2x4',
 'uuid': '5b6ac1de-19dc-4110-85f1-🔐🔐🔐🔐🔐fd51f6b',
 'state': 'Created',
 'time': '2026-02-05T14:39:43Z',
 'projects': []}

In [5]:
osiris.state(job["job"])

{'job': 'training-🔐🔐🔐🔐🔐🔐🔐🔐-pytorchjob-accelerate-fmlib-2x4',
 'uuid': '5b6ac1de-19dc-4110-85f1-🔐🔐🔐🔐🔐fd51f6b',
 'state': 'Created',
 'transition': '2026-02-05T14:39:43Z',
 'time': '2026-02-05T14:39:45Z'}

In [8]:
osiris.logs(job["job"], tail_lines=64, is_master=True)

{'job': 'training-🔐🔐🔐🔐🔐🔐🔐🔐-pytorchjob-accelerate-fmlib-2x4',
 'pods': [{'pod': 'training-🔐🔐🔐🔐🔐🔐🔐🔐-pytorchjob-accelerate-fmlib-2x4-master-0',
   'state': 'Running',
   'logs': ['2026-02-05T17:59:46.🔐🔐🔐🔐🔐🔐🔐🔐🔐+03:00 ',
    'Validation:  58%|█████▊    | 817/1400 [07:25<03:11,  3.04it/s]\x1b[A2026-02-05T17:59:47.🔐🔐🔐🔐🔐🔐🔐🔐🔐+03:00 ',
    '2026-02-05T17:59:47.🔐🔐🔐🔐🔐🔐🔐🔐🔐+03:00 ',
    'Validation:  58%|█████▊    | 818/1400 [07:27<06:01,  1.61it/s]\x1b[A2026-02-05T17:59:47.🔐🔐🔐🔐🔐🔐🔐🔐🔐+03:00 ',
    '2026-02-05T17:59:47.🔐🔐🔐🔐🔐🔐🔐🔐🔐+03:00 ',
    'Validation:  59%|█████▊    | 820/1400 [07:27<03:45,  2.57it/s]\x1b[A2026-02-05T17:59:47.🔐🔐🔐🔐🔐🔐🔐🔐🔐+03:00 ',
    '2026-02-05T17:59:47.🔐🔐🔐🔐🔐🔐🔐🔐🔐+03:00 ',
    'Validation:  59%|█████▊    | 821/1400 [07:27<03:06,  3.11it/s]\x1b[A2026-02-05T17:59:47.🔐🔐🔐🔐🔐🔐🔐🔐🔐+03:00 ',
    '2026-02-05T17:59:47.🔐🔐🔐🔐🔐🔐🔐🔐🔐+03:00 ',
    'Validation:  59%|█████▊    | 822/1400 [07:27<02:34,  3.73it/s]\x1b[A2026-02-05T17:59:48.🔐🔐🔐🔐🔐🔐🔐🔐🔐+03:00 ',
    '2026-02-05T17:59:48.🔐🔐🔐🔐🔐🔐🔐🔐🔐+03:00 ',
    

## Запуск инференса обученной модели

In [ ]:
job = osiris.create(
    name="acc-fmlib-2x4-eval",
    image="registry.ca.sbrf.ru/ci02684173/ci02697916/notebooks/python3.12/cuda12.4/d-03.000.00:d-03.000.00-gigachat",
    restart=False,
    pool="public",
    command=[
        "/home/datalab/nfs/amazmefmlib_hgx_venv/bin/accelerate",
        "launch",
    ],
    args=[
        "--mixed_precision",
        "no",
        "--dynamo_backend",
        "no",
        "--num_machines",
        "2",
        "--num_processes",
        "8",
        "--main_process_ip",
        "$(MASTER_ADDR).datalab.svc.cluster.local",
        "--main_process_port",
        "$(MASTER_PORT)",
        "--machine_rank",
        "$(RANK)",
        "/home/datalab/nfs/amazmefmlib/fmlib/training/evaluate.py",
        "--config-dir=/home/datalab/nfs/amazmefmlib/examples/configs/inference/sequence_representation_2M",
        "--config-name=HGX",
        "+exact_weights_file=/home/datalab/nfs/checkpoints/osiris_test_body_0/checkpoint_000006/model.safetensors",
    ],
    envs={
        "PYTHONPATH": "/home/datalab/nfs/amazmefmlib_hgx_venv",
        "OMP_NUM_THREADS": "24",
        "NCCL_DEBUG": "INFO",
    },
    num_nodes=2,
    num_gpus=4,
    type="pytorchjob",
)
job

In [ ]:
osiris.state(job["job"])

In [ ]:
osiris.logs(job["job"], tail_lines=64, is_master=True)